# 개선된 LSTM 학습 (BiLSTM + Temporal Attention)

> **Runtime → GPU 설정 권장**: Runtime > Change runtime type > T4 GPU

| 개선 사항 | 설명 |
|-----------|------|
| BiLSTM | 양방향 LSTM으로 시퀀스 표현력 강화 |
| Temporal Attention | 중요한 타임스텝에 집중 |
| WeightedRandomSampler | 클래스 불균형 처리 (subsampling 없이 전체 데이터 사용) |
| BCEWithLogitsLoss + pos_weight | 안정적인 손실 함수 |
| AdamW + CosineAnnealingWarmRestarts | 안정적인 학습률 스케줄 |
| Val 기반 threshold 탐색 | 최적 분류 임계값 자동 탐색 |

In [1]:
# [STEP 1] Google Drive 마운트 & 경로 설정
import os, sys

# ★ 본인 Drive 경로에 맞게 수정하세요
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/뉴스 크롤링"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR  = DRIVE_PROJECT_PATH
    DATA_DIR  = os.path.join(BASE_DIR, "")
    MODEL_DIR = os.path.join(BASE_DIR, "models")
    print(f"[Drive 마운트 완료] BASE_DIR={BASE_DIR}")
except Exception:
    BASE_DIR  = os.path.dirname(os.path.abspath('__file__'))
    DATA_DIR  = os.path.join(BASE_DIR, "")
    MODEL_DIR = os.path.join(BASE_DIR, "models")
    print(f"[로컬 환경] BASE_DIR={BASE_DIR}")

os.makedirs(DATA_DIR,  exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print(f"DATA_DIR  : {DATA_DIR}")
print(f"MODEL_DIR : {MODEL_DIR}")
print(f"dataset.npz 존재: {os.path.exists(os.path.join(DATA_DIR, 'dataset1.npz'))}")

Mounted at /content/drive
[Drive 마운트 완료] BASE_DIR=/content/drive/MyDrive/뉴스 크롤링
DATA_DIR  : /content/drive/MyDrive/뉴스 크롤링/
MODEL_DIR : /content/drive/MyDrive/뉴스 크롤링/models
dataset.npz 존재: True


In [2]:
# [STEP 2] 라이브러리 임포트 & 디바이스 설정
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch version : {torch.__version__}")

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"GPU : {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Device : MPS (Apple Silicon)")
else:
    DEVICE = torch.device("cpu")
    print("Device : CPU")
print(f"DEVICE = {DEVICE}")

PyTorch version : 2.10.0+cu128
GPU : Tesla T4
DEVICE = cuda


In [3]:
# [STEP 3] 데이터 로드
path = os.path.join(DATA_DIR, "dataset1.npz")
assert os.path.exists(path), f"dataset1.npz 가 없습니다: {path}"

data    = np.load(path, allow_pickle=True)
X_train = data["X_train"]
X_test  = data["X_test"]
y_train = data["y_train"]
y_test  = data["y_test"]

# 테스트 샘플별 종목코드·날짜 (04_build_dataset.py 재실행 후 생성됨)
test_codes = data["test_codes"] if "test_codes" in data.files else None
test_dates = data["test_dates"] if "test_dates" in data.files else None

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train 상승 비율 : {y_train.mean():.3f}")
print(f"y_test  상승 비율 : {y_test.mean():.3f}")
if test_codes is not None:
    print(f"test_codes 로드 완료: {len(test_codes):,}개, "
          f"종목 수: {len(set(test_codes)):,}, "
          f"날짜 범위: {min(test_dates)} ~ {max(test_dates)}")
else:
    print("[주의] test_codes 없음 → 04_build_dataset.py 재실행 필요")


X_train : (1250163, 20, 18)
X_test  : (317147, 20, 18)
y_train 상승 비율 : 0.477
y_test  상승 비율 : 0.474
test_codes 로드 완료: 317,147개, 종목 수: 836, 날짜 범위: 2019-12-13 ~ 2026-03-31


In [4]:
# [STEP 4] 하이퍼파라미터
HIDDEN_SIZE     = 256    # 128 → 256
NUM_HEADS       = 4      # Multi-head attention
NUM_LAYERS      = 2
DROPOUT         = 0.3
EPOCHS          = 40     # 30 → 40
BATCH_SIZE      = 2048
LR              = 3e-4
PATIENCE        = 10     # 7 → 10
N_ENSEMBLE      = 3      # 앙상블 모델 수
LABEL_SMOOTHING = 0.05   # 레이블 스무딩 (과신 방지)
ENSEMBLE_SEEDS  = [42, 123, 777]

print("하이퍼파라미터 설정 완료")

하이퍼파라미터 설정 완료


In [5]:
# [STEP 5] 모델: BiLSTM + Multi-Head Temporal Attention
class MultiHeadTemporalAttention(nn.Module):
    def __init__(self, hidden_size, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = (hidden_size * 2) // num_heads
        self.attn      = nn.Linear(hidden_size * 2, num_heads)
        self.out_proj  = nn.Linear(hidden_size * 2, hidden_size * 2)

    def forward(self, lstm_out):
        B, T, D = lstm_out.shape
        scores  = self.attn(lstm_out)                                     # (B, T, H)
        weights = F.softmax(scores, dim=1)                                # (B, T, H)
        x       = lstm_out.view(B, T, self.num_heads, self.head_dim)      # (B, T, H, d)
        context = (weights.unsqueeze(-1) * x).sum(dim=1).view(B, D)      # (B, D)
        return self.out_proj(context)


class ImprovedLSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=256, num_layers=2, dropout=0.3, num_heads=4):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_size)
        self.input_proj = nn.Linear(input_size, hidden_size)
        self.lstm = nn.LSTM(
            hidden_size, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True,
        )
        self.attention = MultiHeadTemporalAttention(hidden_size, num_heads)
        self.norm    = nn.LayerNorm(hidden_size * 2)
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(hidden_size * 2, hidden_size)
        self.fc2     = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.input_norm(x)
        x = self.input_proj(x)
        lstm_out, _ = self.lstm(x)
        context = self.attention(lstm_out)
        context = self.norm(context)
        context = self.dropout(context)
        out = F.gelu(self.fc1(context))
        out = self.dropout(out)
        return self.fc2(out).squeeze(-1)


input_size = X_train.shape[2]
_dummy   = ImprovedLSTMClassifier(input_size, HIDDEN_SIZE, NUM_LAYERS, DROPOUT, NUM_HEADS)
n_params = sum(p.numel() for p in _dummy.parameters())
print(f"input_size  : {input_size}")
print(f"파라미터 수 : {n_params:,}")

input_size  : 18
파라미터 수 : 3,031,849


In [6]:
# [STEP 6] 학습 함수
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def find_best_threshold(probs, labels):
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.30, 0.71, 0.02):
        preds = (probs > t).astype(int)
        f1 = f1_score(labels, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    print(f"  최적 threshold={best_t:.2f}  val macro-F1={best_f1:.4f}")
    return best_t


def train_single(X_tr, y_tr, X_val, y_val, X_test, device, seed):
    set_seed(seed)

    # WeightedRandomSampler: 클래스 균형 처리
    # pos_weight는 사용하지 않음 — Sampler와 중복 보정 방지
    class_counts   = np.bincount(y_tr.astype(int))
    sample_weights = np.where(y_tr == 1, class_counts[0] / class_counts[1], 1.0)
    sampler = WeightedRandomSampler(
        weights=torch.FloatTensor(sample_weights),
        num_samples=len(sample_weights), replacement=True,
    )

    pin = (device.type == "cuda")
    train_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr)),
        batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=pin,
    )
    val_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val)),
        batch_size=BATCH_SIZE * 2, shuffle=False,
    )

    model     = ImprovedLSTMClassifier(input_size, HIDDEN_SIZE, NUM_LAYERS, DROPOUT, NUM_HEADS).to(device)
    criterion = nn.BCEWithLogitsLoss()   # pos_weight 제거 — Sampler가 이미 균형 담당

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )

    best_val_loss = float("inf")
    best_state    = None
    no_improve    = 0

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device)
            # Label smoothing: 0 → 0.05, 1 → 0.95 (과신 방지)
            yb_s = (yb * (1 - LABEL_SMOOTHING) + (1 - yb) * LABEL_SMOOTHING).to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb_s)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                yb_s = (yb * (1 - LABEL_SMOOTHING) + (1 - yb) * LABEL_SMOOTHING).to(device)
                val_loss += criterion(model(xb.to(device)), yb_s).item()
        val_loss /= len(val_loader)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            lr_now = optimizer.param_groups[0]["lr"]
            print(f"  [seed={seed}] Epoch {epoch+1:3d}/{EPOCHS}  "
                  f"train={total_loss/len(train_loader):.4f}  "
                  f"val={val_loss:.4f}  lr={lr_now:.1e}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve    = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"  → Early stopping at epoch {epoch+1} (best val={best_val_loss:.4f})")
                break

    model.load_state_dict(best_state)
    model.eval()

    def get_probs(X):
        probs = []
        ld = DataLoader(TensorDataset(torch.FloatTensor(X)), batch_size=BATCH_SIZE * 2)
        with torch.no_grad():
            for (xb,) in ld:
                probs.append(torch.sigmoid(model(xb.to(device))).cpu().numpy())
        return np.concatenate(probs)

    return get_probs(X_val), get_probs(X_test), best_state


print("train_single 함수 정의 완료")

train_single 함수 정의 완료


In [7]:
# [STEP 7] 앙상블 학습 실행 (seed 3개 모델 평균)
n_val        = max(1, int(len(X_train) * 0.15))
X_tr, X_val = X_train[:-n_val], X_train[-n_val:]
y_tr, y_val = y_train[:-n_val], y_train[-n_val:]

print(f"학습={len(X_tr):,}  검증={len(X_val):,}  테스트={len(X_test):,}")
print(f"Up 비율 — 학습: {y_tr.mean():.3f}  검증: {y_val.mean():.3f}")

val_probs_list  = []
test_probs_list = []
best_states     = []

for i, seed in enumerate(ENSEMBLE_SEEDS):
    print(f"\n{'='*45}")
    print(f"  모델 {i+1}/{N_ENSEMBLE}  (seed={seed})")
    print(f"{'='*45}")
    val_p, test_p, state = train_single(X_tr, y_tr, X_val, y_val, X_test, DEVICE, seed)
    val_probs_list.append(val_p)
    test_probs_list.append(test_p)
    best_states.append(state)

# 앙상블 평균 확률로 threshold 탐색 및 최종 예측
avg_val_probs  = np.mean(val_probs_list, axis=0)
avg_test_probs = np.mean(test_probs_list, axis=0)

print("\n[앙상블 threshold 탐색]")
threshold = find_best_threshold(avg_val_probs, y_val)
preds = (avg_test_probs > threshold).astype(int)

save_path = os.path.join(MODEL_DIR, "lstm_improved.pt")
torch.save(best_states[-1], save_path)
print(f"\n모델 저장: {save_path}")

학습=1,062,639  검증=187,524  테스트=317,147
Up 비율 — 학습: 0.478  검증: 0.473

  모델 1/3  (seed=42)
  [seed=42] Epoch   1/40  train=0.6941  val=0.6915  lr=2.9e-04
  [seed=42] Epoch   5/40  train=0.6918  val=0.6923  lr=1.5e-04
  [seed=42] Epoch  10/40  train=0.6908  val=0.6909  lr=3.0e-04
  [seed=42] Epoch  15/40  train=0.6903  val=0.6902  lr=2.6e-04
  [seed=42] Epoch  20/40  train=0.6886  val=0.6914  lr=1.5e-04
  → Early stopping at epoch 22 (best val=0.6900)

  모델 2/3  (seed=123)
  [seed=123] Epoch   1/40  train=0.6934  val=0.6935  lr=2.9e-04
  [seed=123] Epoch   5/40  train=0.6915  val=0.6915  lr=1.5e-04
  [seed=123] Epoch  10/40  train=0.6904  val=0.6913  lr=3.0e-04
  [seed=123] Epoch  15/40  train=0.6900  val=0.6905  lr=2.6e-04
  [seed=123] Epoch  20/40  train=0.6883  val=0.6906  lr=1.5e-04
  [seed=123] Epoch  25/40  train=0.6856  val=0.6930  lr=4.5e-05
  → Early stopping at epoch 27 (best val=0.6902)

  모델 3/3  (seed=777)
  [seed=777] Epoch   1/40  train=0.6937  val=0.6923  lr=2.9e-04
  [seed

In [8]:
# [STEP 8] 평가
acc = accuracy_score(y_test, preds)
f1  = f1_score(y_test, preds, average="weighted")

print("=" * 50)
print("  최종 결과")
print("=" * 50)
print(f"  Accuracy : {acc:.4f}  (기존 LSTM: 0.5082)")
print(f"  F1-score : {f1:.4f}  (기존 LSTM: 0.5055)")
print(f"  Up 예측 비율: {preds.mean():.3f}  (실제: {y_test.mean():.3f})")
print()
print(classification_report(y_test, preds, target_names=["Down", "Up"]))

  최종 결과
  Accuracy : 0.5109  (기존 LSTM: 0.5082)
  F1-score : 0.5002  (기존 LSTM: 0.5055)
  Up 예측 비율: 0.352  (실제: 0.474)

              precision    recall  f1-score   support

        Down       0.53      0.65      0.58    166939
          Up       0.48      0.36      0.41    150208

    accuracy                           0.51    317147
   macro avg       0.50      0.50      0.50    317147
weighted avg       0.50      0.51      0.50    317147



In [9]:
# [STEP 9] predictions.npz 저장
pred_path = os.path.join(DATA_DIR, "predictions.npz")

if os.path.exists(pred_path):
    existing         = dict(np.load(pred_path))
    existing["lstm"] = preds
    np.savez(pred_path, **existing)
    print(f"predictions.npz 업데이트: {pred_path}")
else:
    np.savez(pred_path, lstm=preds, y_test=y_test)
    print(f"predictions.npz 새로 저장: {pred_path}")

print(f"Down={( preds==0).sum():,}  Up={(preds==1).sum():,}")

predictions.npz 업데이트: /content/drive/MyDrive/뉴스 크롤링/predictions.npz
Down=205,423  Up=111,724


In [10]:
# [STEP 10] 확률값 준비 (학습 없이 저장된 모델로 추론)
# - STEP 7이 이미 실행된 세션이면 avg_test_probs 가 메모리에 있음 → 그대로 사용
# - 새 세션(런타임 재시작)이면 저장된 모델을 로드해 재추론

def get_probs_from_model(model, X, batch_size, device):
    model.eval()
    probs = []
    loader = DataLoader(TensorDataset(torch.FloatTensor(X)),
                        batch_size=batch_size * 2, shuffle=False)
    with torch.no_grad():
        for (xb,) in loader:
            probs.append(torch.sigmoid(model(xb.to(device))).cpu().numpy())
    return np.concatenate(probs)


if 'avg_test_probs' in dir():
    lstm_test_probs = avg_test_probs
    print("STEP 7 결과 재사용 (앙상블 평균 확률)")
else:
    print("저장된 모델 로드 후 재추론 중...")
    _model = ImprovedLSTMClassifier(
        X_test.shape[2], HIDDEN_SIZE, NUM_LAYERS, DROPOUT, NUM_HEADS
    ).to(DEVICE)
    _state = torch.load(
        os.path.join(MODEL_DIR, 'lstm_improved.pt'),
        map_location=DEVICE
    )
    _model.load_state_dict(_state)
    lstm_test_probs = get_probs_from_model(_model, X_test, BATCH_SIZE, DEVICE)
    print("재추론 완료")

# 확률값을 predictions.npz에 저장해두어 이후 재활용 가능하게
_pred_path = os.path.join(DATA_DIR, 'predictions.npz')
_existing  = dict(np.load(_pred_path))
_existing['lstm_probs'] = lstm_test_probs
np.savez(_pred_path, **_existing)

print(f"lstm_probs shape : {lstm_test_probs.shape}")
print(f"prob 범위 : [{lstm_test_probs.min():.3f}, {lstm_test_probs.max():.3f}]")
print(f"prob 평균 : {lstm_test_probs.mean():.3f}")
print("predictions.npz 에 lstm_probs 저장 완료")

STEP 7 결과 재사용 (앙상블 평균 확률)
lstm_probs shape : (317147,)
prob 범위 : [0.294, 0.905]
prob 평균 : 0.505
predictions.npz 에 lstm_probs 저장 완료


In [ ]:
# [STEP 11] Threshold별 Up Precision / Recall / Coverage 분석
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

# lstm_test_probs가 없으면 predictions.npz에서 로드 (런타임 재시작 시 대비)
if 'lstm_test_probs' not in dir():
    _p = np.load(os.path.join(DATA_DIR, 'predictions.npz'))
    lstm_test_probs = _p['lstm_probs']
    print(f"predictions.npz에서 lstm_probs 로드 완료 (shape={lstm_test_probs.shape})")
else:
    print("기존 lstm_test_probs 변수 사용")

y = y_test.astype(int)
thresholds = np.arange(0.40, 0.76, 0.02)

rows = []
for t in thresholds:
    preds_t  = (lstm_test_probs > t).astype(int)
    n_up     = int(preds_t.sum())
    coverage = float(preds_t.mean())
    if n_up == 0:
        rows.append(dict(threshold=round(t,2), precision=0, recall=0,
                         f1_up=0, coverage=0, n_up=0))
        continue
    rows.append(dict(
        threshold = round(t, 2),
        precision = precision_score(y, preds_t, pos_label=1, zero_division=0),
        recall    = recall_score(y, preds_t, pos_label=1, zero_division=0),
        f1_up     = f1_score(y, preds_t, pos_label=1, zero_division=0),
        coverage  = coverage,
        n_up      = n_up,
    ))

df_thr = pd.DataFrame(rows)
print(df_thr.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# 시각화
fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.plot(df_thr['threshold'], df_thr['precision'], 'o-',  color='#DD8452', label='Up Precision')
ax1.plot(df_thr['threshold'], df_thr['recall'],    's--', color='#4C72B0', label='Up Recall')
ax1.plot(df_thr['threshold'], df_thr['f1_up'],     '^:',  color='#55A868', label='Up F1')
ax2.bar(df_thr['threshold'], df_thr['coverage'], width=0.015,
        alpha=0.25, color='gray', label='Coverage')
ax1.axhline(0.5, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Random baseline')
ax1.set_xlabel('Threshold')
ax1.set_ylabel('Score')
ax1.set_ylim(0, 1)
ax2.set_ylabel('Coverage (Up 예측 비율)')
ax2.set_ylim(0, 1)
ax1.set_title('LSTM: Up Precision / Recall / F1 vs Threshold')
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='center left')
plt.tight_layout()
os.makedirs(os.path.join(DATA_DIR, 'results'), exist_ok=True)
plt.savefig(os.path.join(DATA_DIR, 'results', 'lstm_threshold_curve.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# ── threshold 선택 후 예측 확정 ──
CHOSEN_THRESHOLD = 0.60  # ★ 표를 보고 원하는 값으로 조정

final_preds = (lstm_test_probs > CHOSEN_THRESHOLD).astype(int)
print(f"\n[threshold={CHOSEN_THRESHOLD}] 최종 적용")
print(classification_report(y, final_preds, target_names=['Down', 'Up']))

_existing = dict(np.load(os.path.join(DATA_DIR, 'predictions.npz')))
_existing['lstm'] = final_preds
np.savez(os.path.join(DATA_DIR, 'predictions.npz'), **_existing)
print(f"predictions.npz lstm 업데이트  "
      f"(Down={(final_preds==0).sum():,}  Up={(final_preds==1).sum():,})")

# ── 날짜별 상승 예측 종목 조회 ──
if test_codes is not None:
    df_pred = pd.DataFrame({
        'code':      test_codes,
        'date':      test_dates,
        'prob':      lstm_test_probs,
        'pred':      final_preds,
        'actual':    y,
    })

    latest_date = df_pred['date'].max()
    TOP_N = 100

    df_today = (
        df_pred[df_pred['date'] == latest_date]
        .sort_values('prob', ascending=False)
    )
    up_today = df_today[df_today['pred'] == 1]

    print(f"\n[{latest_date}] threshold={CHOSEN_THRESHOLD} 기준 상승 예측 종목: {len(up_today)}개")
    print(up_today[['code', 'prob', 'actual']].head(TOP_N).to_string(index=False))

    out_csv = os.path.join(DATA_DIR, 'results', f'up_picks_{latest_date}.csv')
    up_today.to_csv(out_csv, index=False, encoding='utf-8-sig')
    print(f"저장: {out_csv}")
else:
    print("[주의] test_codes 없음 → 04_build_dataset.py 재실행 후 dataset.npz 업데이트 필요")

# [STEP 12] LightGBM 비교 실험

| | LSTM (BiLSTM+Attention) | LightGBM |
|--|--|--|
| 구조 | 딥러닝 시계열 모델 | 트리 기반 앙상블 |
| 학습 시간 | ~30분 | ~5분 |
| 특징 | 순서 패턴 자동 학습 | 조건 조합 분기에 강함 |

LSTM과 성능을 비교하여 모델 선택의 타당성을 분석합니다.

In [ ]:
# [STEP 12] LightGBM + Walk-forward Cross-Validation
import lightgbm as lgb
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
import pandas as pd

# ── 공통 하이퍼파라미터 ───────────────────────────────────────────────────────
# [수정 근거]
#   1. class_weight="balanced" 제거:
#      Up 비율이 ~54%로 다수 클래스이므로, balanced가 오히려 Up에 낮은 가중치를 부여 → recall 저하
#   2. scale_pos_weight=1.5 → 1.2:
#      1.5에서 2026-03 recall=1.0 (전부 Up 예측) 발생 → 과도한 Up 편향 완화
#   3. num_leaves 31→63: 더 복잡한 조건 분기 학습 (표현력 ↑)
#   4. min_child_samples 50→20: 보수적 분기 완화 → recall 개선
#   5. reg_alpha 0.1→0.05, reg_lambda 1.0→0.5: 정규화 완화

LGB_PARAMS = dict(
    n_estimators      = 800,
    learning_rate     = 0.02,
    num_leaves        = 63,        # ↑ 31→63
    min_child_samples = 20,        # ↓ 50→20
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.05,      # ↓ 0.1→0.05
    reg_lambda        = 0.5,       # ↓ 1.0→0.5
    scale_pos_weight  = 1.2,       # ↓ 1.5→1.2: recall 과잉 억제 (2026-03 recall=1.0 방지)
    random_state      = 42,
    n_jobs            = -1,
    verbose           = -1,
)

# ── Walk-forward CV ───────────────────────────────────────────────────────────
_train_dates = data["train_dates"] if "train_dates" in data.files else None
_test_dates  = data["test_dates"]  if "test_dates"  in data.files else None

WF_THRESHOLD = 0.45   # 0.50→0.45: threshold 분석 결과 F1 최적점

if _train_dates is None:
    print("[경고] train_dates 없음 → 04_build_dataset.ipynb 재실행 필요")
else:
    _all_X     = np.concatenate([
                     X_train.reshape(len(X_train), -1),
                     X_test.reshape(len(X_test), -1),
                 ])
    _all_y     = np.concatenate([y_train, y_test])
    _all_dates = pd.to_datetime(np.concatenate([_train_dates, _test_dates]))

    all_months   = _all_dates.to_period("M").sort_values().unique()
    MIN_TRAIN_MO = 6
    eval_months  = all_months[MIN_TRAIN_MO:]

    print(f"전체 샘플: {len(_all_X):,}  |  평가 폴드 수: {len(eval_months)}")
    print(f"평가 기간: {eval_months[0]} ~ {eval_months[-1]}")
    print(f"Walk-forward threshold: {WF_THRESHOLD}\n")

    fold_rows = []
    for fold_month in eval_months:
        fold_start = fold_month.to_timestamp()
        fold_end   = (fold_month + 1).to_timestamp()
        tr_mask    = _all_dates < fold_start
        te_mask    = (_all_dates >= fold_start) & (_all_dates < fold_end)

        if tr_mask.sum() < 500 or te_mask.sum() < 20:
            continue

        X_tr_f, y_tr_f = _all_X[tr_mask], _all_y[tr_mask]
        X_te_f, y_te_f = _all_X[te_mask], _all_y[te_mask]

        n_val = max(1, int(len(X_tr_f) * 0.1))
        clf   = lgb.LGBMClassifier(**LGB_PARAMS)
        clf.fit(
            X_tr_f[:-n_val], y_tr_f[:-n_val],
            eval_set=[(X_tr_f[-n_val:], y_tr_f[-n_val:])],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
        )

        probs_f = clf.predict_proba(X_te_f)[:, 1]
        preds_f = (probs_f > WF_THRESHOLD).astype(int)
        prec = precision_score(y_te_f, preds_f, pos_label=1, zero_division=0)
        rec  = recall_score(y_te_f,    preds_f, pos_label=1, zero_division=0)
        f1   = f1_score(y_te_f,        preds_f, pos_label=1, zero_division=0)
        acc  = (preds_f == y_te_f.astype(int)).mean()

        fold_rows.append(dict(fold=str(fold_month), n_train=int(tr_mask.sum()),
                              n_test=int(te_mask.sum()), accuracy=round(acc,4),
                              precision=round(prec,4), recall=round(rec,4), f1_up=round(f1,4)))
        print(f"  {fold_month} | train={tr_mask.sum():>7,}  test={te_mask.sum():>5,} "
              f"| acc={acc:.3f}  prec={prec:.3f}  rec={rec:.3f}  f1={f1:.3f}")

    df_wf = pd.DataFrame(fold_rows)
    print(f"\n{'='*55}")
    print("  Walk-forward 평균 성능")
    print(f"{'='*55}")
    print(df_wf[["accuracy","precision","recall","f1_up"]].mean().round(4).to_string())
    # 경고: 특정 폴드에서 recall≈1.0이면 모델 퇴화(전부 Up 예측) 의심
    degenerate = df_wf[df_wf["recall"] > 0.95]
    if len(degenerate) > 0:
        print(f"\n⚠ recall>0.95 폴드 (모델 퇴화 의심): {degenerate['fold'].tolist()}")
        print("  → scale_pos_weight를 더 낮추거나 WF_THRESHOLD를 올리세요")
    print()

# ── 단일 분할 LightGBM ──────────────────────────────────────────────────────
print("── 단일 분할 LightGBM ─────────────────────────────────────────────")
n_val_lgb  = max(1, int(len(X_train) * 0.15))
X_tr_lgb   = X_train[:-n_val_lgb].reshape(len(X_train) - n_val_lgb, -1)
X_val_lgb  = X_train[-n_val_lgb:].reshape(n_val_lgb, -1)
X_te_lgb   = X_test.reshape(len(X_test), -1)
y_tr_lgb   = y_train[:-n_val_lgb]
y_val_lgb  = y_train[-n_val_lgb:]

print(f"학습: {X_tr_lgb.shape}  검증: {X_val_lgb.shape}")
print(f"Up 비율 — 학습: {y_tr_lgb.mean():.3f}  검증: {y_val_lgb.mean():.3f}\n")

model_lgb = lgb.LGBMClassifier(**LGB_PARAMS, n_estimators=1000)
model_lgb.fit(
    X_tr_lgb, y_tr_lgb,
    eval_set=[(X_val_lgb, y_val_lgb)],
    callbacks=[lgb.early_stopping(100, verbose=True), lgb.log_evaluation(100)],
)

lgb_probs    = model_lgb.predict_proba(X_te_lgb)[:, 1]
lgb_preds_50 = (lgb_probs > 0.50).astype(int)
lgb_preds_45 = (lgb_probs > 0.45).astype(int)

print("\n" + "="*50)
print("  LightGBM 결과 (threshold=0.50)")
print("="*50)
print(classification_report(y_test.astype(int), lgb_preds_50, target_names=["Down","Up"]))

print("="*50)
print("  LightGBM 결과 (threshold=0.45)")
print("="*50)
print(classification_report(y_test.astype(int), lgb_preds_45, target_names=["Down","Up"]))

# Threshold 분석
thresholds = np.arange(0.40, 0.76, 0.02)
rows_lgb = []
for t in thresholds:
    p_t  = (lgb_probs > t).astype(int)
    n_up = int(p_t.sum())
    if n_up == 0:
        rows_lgb.append(dict(threshold=round(t,2), precision=0, recall=0, f1_up=0, coverage=0, n_up=0))
        continue
    rows_lgb.append(dict(
        threshold = round(t, 2),
        precision = precision_score(y_test.astype(int), p_t, pos_label=1, zero_division=0),
        recall    = recall_score(y_test.astype(int),    p_t, pos_label=1, zero_division=0),
        f1_up     = f1_score(y_test.astype(int),        p_t, pos_label=1, zero_division=0),
        coverage  = float(p_t.mean()),
        n_up      = n_up,
    ))

df_lgb_thr = pd.DataFrame(rows_lgb)
print("\n[LightGBM Threshold 분석]")
print(df_lgb_thr.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

best_idx = df_lgb_thr["f1_up"].idxmax()
best_row = df_lgb_thr.loc[best_idx]
print(f"\n★ F1 최적 threshold: {best_row['threshold']:.2f}  "
      f"(prec={best_row['precision']:.3f}, rec={best_row['recall']:.3f}, f1={best_row['f1_up']:.3f})")

In [ ]:
# [STEP 12-2] Walk-forward 결과 & LightGBM 확률값 저장
import os

# Walk-forward 결과 CSV 저장 (07_visualization.ipynb에서 로드)
if 'df_wf' in dir() and len(df_wf) > 0:
    wf_path = os.path.join(DATA_DIR, "results", "lgb_walkforward.csv")
    os.makedirs(os.path.join(DATA_DIR, "results"), exist_ok=True)
    df_wf.to_csv(wf_path, index=False, encoding="utf-8-sig")
    print(f"Walk-forward 결과 저장: {wf_path}")
    print(df_wf.to_string(index=False))
else:
    print("[주의] df_wf 없음 → train_dates가 저장된 dataset.npz 필요")

# LightGBM 예측 결과 predictions.npz에 저장
if 'lgb_probs' in dir() and 'model_lgb' in dir():
    _pred_path = os.path.join(DATA_DIR, "predictions.npz")
    _existing  = dict(np.load(_pred_path))
    _existing["lgb"]       = lgb_preds_50
    _existing["lgb_probs"] = lgb_probs
    np.savez(_pred_path, **_existing)
    print(f"\npredictions.npz 업데이트: lgb, lgb_probs 저장 완료")
    print(f"  Down={(lgb_preds_50==0).sum():,}  Up={(lgb_preds_50==1).sum():,}")


In [ ]:
# [STEP 12-1] LightGBM Feature Importance 분석
import matplotlib.pyplot as plt
import pandas as pd

# dataset.npz에서 feature 이름 로드 (없으면 직접 정의)
if 'feature_cols' in data.files:
    feature_cols = list(data['feature_cols'])
else:
    feature_cols = [
        "log_return", "volume",
        "sma_5", "sma_20", "sma_60",
        "rsi", "macd", "macd_signal", "macd_hist",
        "bb_upper", "bb_lower", "bb_width",
        "return_5d", "return_20d",
        "volume_ratio_20", "high_52w_ratio", "volatility_20",
        "relative_return",
        "sentiment_mean_weighted", "sentiment_std_weighted",
        "sentiment_lag1", "sentiment_lag2", "sentiment_change",
        "news_count_zscore_20", "has_news",
    ]

n_feats  = len(feature_cols)
n_steps  = X_train.shape[1]   # 20

# 플랫 feature 이름: feature_t{step} 형태
flat_names = [f"{feature_cols[i % n_feats]}_t{i // n_feats}"
              for i in range(n_feats * n_steps)]

importance_df = pd.DataFrame({
    "flat_name": flat_names,
    "feature"  : [feature_cols[i % n_feats] for i in range(n_feats * n_steps)],
    "importance": model_lgb.feature_importances_,
})

# timestep 합산 → 원래 feature 단위 중요도
agg = (importance_df.groupby("feature")["importance"]
       .sum()
       .sort_values(ascending=False)
       .reset_index())

print("\n[Feature 중요도 (전체 timestep 합산)]")
print(agg.to_string(index=False))

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 왼쪽: 원래 feature 단위
agg.set_index("feature")["importance"].plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("Feature Importance (aggregated)")
axes[0].set_ylabel("Importance")
axes[0].tick_params(axis="x", rotation=45)

# 오른쪽: 상위 20개 flat feature (어느 timestep이 중요한지)
top20 = importance_df.nlargest(20, "importance")
top20.set_index("flat_name")["importance"].plot(kind="barh", ax=axes[1], color="#DD8452")
axes[1].set_title("Top 20 Flat Features (feature × timestep)")
axes[1].invert_yaxis()

plt.tight_layout()
os.makedirs(os.path.join(DATA_DIR, "results"), exist_ok=True)
plt.savefig(os.path.join(DATA_DIR, "results", "lgb_feature_importance.png"),
            dpi=150, bbox_inches="tight")
plt.show()

# 기여 없는 feature (importance=0) 확인
zero_feats = agg[agg["importance"] == 0]["feature"].tolist()
if zero_feats:
    print(f"\n⚠ importance=0 feature: {zero_feats}")
    print("→ 04_build_dataset.ipynb의 FEATURE_COLS에서 제거 고려")
else:
    print("\n모든 feature가 사용됨")

In [ ]:
# [STEP 13] LightGBM predictions.npz 저장 & 날짜별 종목 조회
LGB_THRESHOLD = 0.60  # ★ 원하는 값으로 조정

lgb_final_preds = (lgb_probs > LGB_THRESHOLD).astype(int)

_pred_path = os.path.join(DATA_DIR, 'predictions.npz')
_existing  = dict(np.load(_pred_path))
_existing['lgb']       = lgb_final_preds
_existing['lgb_probs'] = lgb_probs
np.savez(_pred_path, **_existing)
print(f"predictions.npz 업데이트 (lgb, lgb_probs 추가)")
print(f"Down={(lgb_final_preds==0).sum():,}  Up={(lgb_final_preds==1).sum():,}")

# ── 날짜별 상승 예측 종목 조회 ──
if test_codes is not None:
    df_lgb_pred = pd.DataFrame({
        'code'  : test_codes,
        'date'  : test_dates,
        'prob'  : lgb_probs,
        'pred'  : lgb_final_preds,
        'actual': y_test.astype(int),
    })

    latest_date = df_lgb_pred['date'].max()
    df_today_lgb = df_lgb_pred[df_lgb_pred['date'] == latest_date].sort_values('prob', ascending=False)
    up_today_lgb = df_today_lgb[df_today_lgb['pred'] == 1]

    print(f"\n[{latest_date}] LightGBM threshold={LGB_THRESHOLD} 기준 상승 예측 종목: {len(up_today_lgb)}개")
    print(up_today_lgb[['code', 'prob', 'actual']].head(50).to_string(index=False))

    out_csv = os.path.join(DATA_DIR, 'results', f'lgb_up_picks_{latest_date}.csv')
    up_today_lgb.to_csv(out_csv, index=False, encoding='utf-8-sig')
    print(f"저장: {out_csv}")